In [1]:
import torch
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from pytorch3d.loss import chamfer_distance
from pathlib import Path
from dataset import *
from decoder import *
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from visualization import scatter_vis
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter

In [2]:
preprocessing_path = "../data/preprocessed/preprocessing6"
test_path = "../data/preprocessed/test"
model_path = "./models/Jun_Sun_16_14_17/end_model.pt"

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

hp = {
    "latent_dim": 256,
    "num_steps": 200,
    "beta_1": 1e-4,
    "beta_T": 5e-2,
    "num_points": 512,
    "hidden_dim": 128,
}

if Path(preprocessing_path).exists:
    print("Preprocessing Path exists")

if Path(test_path).exists:
    print("Test Path exists")
    
if Path(model_path).exists:
    print("Model Path exists")

model = AutoEncoder(number_points=512, point_dim=3, hidden_dim=128, latent_dim=hp["latent_dim"], num_steps=200,beta_1=0.0001, beta_T=0.05, kl_start=0.0001)
state_dict = torch.load(model_path, map_location=device)
#print(model.state_dict)
#print(state_dict)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

Preprocessing Path exists
Test Path exists
Model Path exists


AutoEncoder(
  (encoder): PointNetEncoder(
    (projection): Sequential(
      (0): Conv1d(3, 256, kernel_size=(1,), stride=(1,))
      (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU()
      (3): Conv1d(256, 256, kernel_size=(1,), stride=(1,))
      (4): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (5): ReLU()
      (6): Conv1d(256, 512, kernel_size=(1,), stride=(1,))
      (7): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (8): Conv1d(512, 1024, kernel_size=(1,), stride=(1,))
    )
    (max_pool): MaxPool1d(kernel_size=512, stride=512, padding=0, dilation=1, ceil_mode=False)
    (fc1_mean): Linear(in_features=1024, out_features=512, bias=True)
    (fc2_mean): Linear(in_features=512, out_features=256, bias=True)
    (fc3_mean): Linear(in_features=256, out_features=256, bias=True)
    (fc1_mean_norm): BatchNorm1d(512, 

In [3]:
preprocessing_point_clouds_ds = PointCloudDataset(get_point_cloud_files(preprocessing_path), number_points=512)
test_point_clouds_ds = PointCloudDataset(get_point_cloud_files(test_path), number_points=512)

point_clouds = DataLoader(preprocessing_point_clouds_ds, batch_size=10, drop_last=True)
test_point_clouds = DataLoader(test_point_clouds_ds, batch_size=10, drop_last=True)

In [4]:
def chamfer_custom(pc_a, pc_b):
    # pc_a: [B, N, 3]
    # pc_b: [B, M, 3]

    dists = torch.cdist(pc_a, pc_b, p=2) ** 2
    # shape: [B, N, M]

    pc_a_to_pc_b = dists.min(dim=2).values  # [B, N]
    pc_b_to_pc_a = dists.min(dim=1).values  # [B, M]

    return pc_a_to_pc_b.mean() + pc_b_to_pc_a.mean()

In [25]:
chamfer_loss_total = 0
for batch in test_point_clouds:
    _, means, _ = model.encode(batch)
    pred_pcs = model.decode(means, num_points=512)
    chamfer_loss =  chamfer_custom(pred_pcs, batch)
    chamfer_loss_total += chamfer_loss.sum(dim=0)
print(f"chamfer loss: {chamfer_loss_total / len(test_point_clouds)}")

chamfer loss: 0.08394695818424225


In [18]:
chamfer_loss_total = 0
for batch in test_point_clouds:
    _, means, _ = model.encode(batch)
    pred_pcs = model.decode(means, num_points=512)
    chamfer_loss, _ =  chamfer_distance(pred_pcs, batch)
    chamfer_loss_total += chamfer_loss.sum(dim=0)
print(f"chamfer loss: {chamfer_loss_total / len(test_point_clouds)}")

chamfer loss: 0.08876916766166687


In [5]:
batch = next(iter(test_point_clouds))
_, means, _ = model.encode(batch)
pred_pcs = model.decode(means, num_points=512)
%matplotlib qt
target_pcs1 = []
target_pcs2 = []
target_pcs3 = []

for i in range(3):
    target_pcs1.append(batch[i])
    target_pcs2.append(batch[i+3])
    target_pcs3.append(batch[i+6])
    
scatter_vis(pred_pcs, target_pcs1)
scatter_vis(pred_pcs[3:], target_pcs2)
scatter_vis(pred_pcs[6:], target_pcs3)

In [27]:
point_cloud_means_np = []
test_point_cloud_np = []
for batch in point_clouds:
    _, point_clouds_means, _ = model.encode(batch)
    point_cloud_means_np.append(point_clouds_means.detach().cpu().numpy())

for batch in test_point_clouds:
    _, test_point_cloud_means, _ = model.encode(batch)
    test_point_cloud_np.append(test_point_cloud_means.detach().cpu().numpy())

point_cloud_means_np = np.stack(point_cloud_means_np, axis=0).reshape(-1, hp["latent_dim"])
test_point_cloud_np = np.stack(test_point_cloud_np, axis=0).reshape(-1, hp["latent_dim"])
points_combined_np = np.vstack((point_cloud_means_np, test_point_cloud_np))
print(point_cloud_means_np.shape)
print(test_point_cloud_np.shape)
print(points_combined_np.shape)    

(15380, 256)
(150, 256)
(15530, 256)


In [12]:
%matplotlib qt
cov = np.cov(points_combined_np.T)
print(cov.shape)
eig, _ = np.linalg.eig(cov)
eig_cum_sum = np.cumsum(eig)
eig_percentiles = [0] + (eig_cum_sum / np.sum(eig)).tolist()
plt.title("Explainability of Variance")
plt.grid()
#plt.xlim([0, 130])
#plt.xticks(range(0, 131, 10))
plt.xlabel("Size of latent space \u2192")
plt.ylabel("Explainability of Variance \u2192")
plt.plot(range(len(eig_percentiles)), eig_percentiles)
plt.show()

(32, 32)


In [28]:
t_sne = TSNE(n_components=2, perplexity=30, init="random", max_iter=250, random_state=0)
points_combined_tsne = t_sne.fit_transform(points_combined_np)

fig = plt.figure()
plt.grid()
plt.title("Visualization of Latent Space in 2d t-sne")
plt.xlabel("x \u2192")
plt.ylabel("y \u2192")
plt.scatter(points_combined_tsne[:, 0], points_combined_tsne[:, 1], s=0.5, label="Latent Vectors projected to 2d")
plt.legend()
plt.show()

In [29]:
pca = PCA(n_components=2)
pca.fit(points_combined_np)
points_mean_pca = pca.transform(point_cloud_means_np)
test_points_mean_pca = pca.transform(test_point_cloud_np)

fig = plt.figure()
plt.grid()
plt.title("Visualization of latent Space in 2d pca")
plt.xlabel("x \u2192")
plt.ylabel("y \u2192")
plt.scatter(points_mean_pca[:, 0], points_mean_pca[:, 1], s=0.5, color="b", label="Latent Vectors of training set projected to 2d")
plt.scatter(test_points_mean_pca[:, 0], test_points_mean_pca[:, 1], s=10, color="r", label="Latent Vectors of test set projected to 2d")
plt.legend()
plt.show()

In [30]:
@torch.no_grad
def reconstruction_from_random(model):
    latents = torch.randn((10, hp["latent_dim"]))
    pcs_from_random = model.decode(latents, 1024)
    
    for i in range(len(pcs_from_random)):
        pc = pcs_from_random[i].detach().cpu().numpy()
        fig = plt.figure(figsize=(12, 12))
        ax = fig.add_subplot(1, 2, 1, projection="3d")
        ax.set_axis_off()
        ax.grid("off")
        ax.set_title("Point Cloud reconstruction from random latent")
        ax.scatter(pc[:, 0], pc[:, 1], pc[:, 2])
        
        latent = latents[i].unsqueeze(dim=0).detach().cpu().numpy()
        random_latent_pca = pca.transform(latent)
        ax = fig.add_subplot(1, 2, 2)
        ax.set_title("Visualization of Latent Space in 2d pca")
        ax.scatter(points_mean_pca[:, 0], points_mean_pca[:, 1], color="b", s=1, label="Latent Vectors of training set projected to 2d")
        ax.scatter(random_latent_pca[:, 0], random_latent_pca[:, 1], color="r", s=50, label="Random Reconstruction Latent Vector projected to 2d")
        plt.legend()
        plt.tight_layout()
        plt.show()

reconstruction_from_random(model)

In [31]:
latent_space_inp = "../data/preprocessed/latent_space"
latent_space_inp = PointCloudDataset(get_point_cloud_files(latent_space_inp), number_points=512)
latent_space_inp = DataLoader(latent_space_inp, batch_size=2)

In [32]:
@torch.no_grad
def interpolate_point_clouds(model, batch, num_frames):
    model.eval()
    
    pc_a = batch[0].to(device)
    pc_b = batch[1].to(device)
    
    if pc_a.dim() == 2:
        pc_a = pc_a.unsqueeze(dim=0)
    if pc_b.dim() == 2:
        pc_b = pc_b.unsqueeze(dim=0)
    
    _, latent_a, _ = model.encode(pc_a)
    _, latent_b, _ = model.encode(pc_b)
    
    alpha = torch.linspace(0, 1, num_frames)
    
    point_clouds = []
    
    for i in range(len(alpha)):
        z = latent_a * (1- alpha[i]) + latent_b * alpha[i]
        pc = model.decode(z, num_points=1024)
        pc = pc.squeeze(0).cpu().numpy()
        point_clouds.append(pc)
    
    return point_clouds


def get_axis_limits(pointclouds, margin=0.05):
    all_points = np.concatenate(pointclouds, axis=0)

    mins = all_points.min(axis=0)
    maxs = all_points.max(axis=0)

    center = (mins + maxs) / 2
    scale = (maxs - mins).max() / 2
    scale *= 1 + margin

    return (
        (center[0] - scale, center[0] + scale),
        (center[1] - scale, center[1] + scale),
        (center[2] - scale, center[2] + scale),
    )


def plot_interpolation_grid(pointclouds, cols=5, figsize=(15, 8)):
    rows = int(np.ceil(len(pointclouds) / cols))

    fig = plt.figure(figsize=figsize)
    xlim, ylim, zlim = get_axis_limits(pointclouds)

    for i, pc in enumerate(pointclouds):
        ax = fig.add_subplot(rows, cols, i + 1, projection="3d")

        ax.scatter(pc[:, 0], pc[:, 1], pc[:, 2], s=1.5)
        
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_zlim(zlim)

        ax.set_title(f"alpha = {i / (len(pointclouds) - 1):.2f}")
        ax.set_axis_off()

    plt.tight_layout()
    plt.show()
    

def animate_pointclouds(
    pointclouds,
    interval=100,
    save_path=None,
    fps=15,
    elev=20,
    azim=45,
):
    fig = plt.figure(figsize=(12, 12))
    ax = fig.add_subplot(111, projection="3d")
    
    xlim, ylim, zlim = get_axis_limits(pointclouds)

    first_pc = pointclouds[0]

    scatter = ax.scatter(
        first_pc[:, 0],
        first_pc[:, 1],
        first_pc[:, 2],
        s=2,
    )
    
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_zlim(zlim)


    ax.view_init(elev=elev, azim=azim)
    ax.set_axis_off()

    title = ax.set_title("alpha = 0.00")

    def update(frame_idx):
        pc = pointclouds[frame_idx]

        scatter._offsets3d = (
            pc[:, 0],
            pc[:, 1],
            pc[:, 2],
        )

        alpha = frame_idx / (len(pointclouds) - 1)
        title.set_text(f"alpha = {alpha:.2f}")

        return scatter, title

    anim = FuncAnimation(
        fig,
        update,
        frames=len(pointclouds),
        interval=interval,
        blit=False,
    )

    if save_path is not None:
        if save_path.endswith(".gif"):
            anim.save(save_path, writer=PillowWriter(fps=fps))
        elif save_path.endswith(".mp4"):
            anim.save(save_path, writer=FFMpegWriter(fps=fps))
        else:
            raise ValueError("save_path must end with .gif or .mp4")

    plt.show()

    return anim

In [33]:
latent_space_inp = iter(latent_space_inp)
batch1 = next(latent_space_inp)
batch2 = next(latent_space_inp)
batch3 = next(latent_space_inp)

In [34]:
point_cloud_set1 = interpolate_point_clouds(model, batch2, 100)
#plot_interpolation_grid(point_cloud_set1)
animate_pointclouds(point_cloud_set1, interval=100, save_path="./models/Jun_Sun_16_14_17/LatentSpaceInterpolation2.gif")